In [ ]:
#setup
!pip install telethon==1.36.0 confluent-kafka
!pip install --upgrade confluent-kafka

In [ ]:
#setup a local sql table to save messages
import sqlite3

def init_db(db_path="messages.db"):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            channel TEXT,
            sender TEXT,
            text TEXT,
            ts INTEGER,
            PRIMARY KEY (channel, ts, sender, text)
        )
    """)
    conn.commit()
    conn.close()
    print("Database ready: messages.db")

init_db()

In [ ]:
#telegram scraper
import json
import sqlite3
import asyncio
from telethon import TelegramClient

# Load settings from config.json
with open("config.json", "r") as f:
    config = json.load(f)

API_ID = config["api_id"]
API_HASH = config["api_hash"]
SESSION_NAME = config.get("session_name", "scraper_session")
CHANNELS = config["channels"]

async def scrape_history(limit_per_channel=5000):
    client = TelegramClient(SESSION_NAME, API_ID, API_HASH)
    await client.start()
    
    conn = sqlite3.connect("messages.db")
    cur = conn.cursor()
    
    for ch in CHANNELS:
        print(f"Scraping {ch}...")
        count = 0
        async for msg in client.iter_messages(ch, limit=limit_per_channel):
            if not msg.text:
                continue
            
            sender = str(msg.sender_id or "unknown")
            ts = int(msg.date.timestamp())
            
            cur.execute("""
                INSERT OR IGNORE INTO messages (channel, sender, text, ts)
                VALUES (?, ?, ?, ?)
            """, (ch, sender, msg.text, ts))
            count += 1
            
        conn.commit()
        print(f"Saved {count} messages from {ch}")
        
    conn.close()
    await client.disconnect()
    print("Scraping complete!")

# Run in Jupyter:
await scrape_history(limit_per_channel=3000)